In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
project_root = Path().cwd().parent / "data" / "cleaned_data"

In [3]:
Customer_data = pd.read_csv(project_root / "customer_master_cleaned.csv")
Ecommerce_data = pd.read_csv(project_root / "ecommerce_sales_customer_analytics_150k_cleaned.csv")
Order_data = pd.read_csv(project_root / "order_items_cleaned.csv")
Product_data = pd.read_csv(project_root / "product_catalog_cleaned.csv")

In [4]:
Customer_data.head()
print(Customer_data['customer_id'].duplicated().sum())
print(Ecommerce_data['order_id'].duplicated().sum())
print(Order_data['order_id'].duplicated().sum())
print(Product_data['product_id'].duplicated().sum())
print(Ecommerce_data['customer_id'].duplicated().sum())
print(Order_data['product_id'].duplicated().sum())

0
0
259366
0
113205
396307


In [5]:
# Total rows and unique orders
print("Total order-item rows:", len(Order_data))
print("Unique order IDs:", Order_data["order_id"].nunique())
print("Repeated order ID rows:", Order_data["order_id"].duplicated().sum())

# Show orders that appear more than once
print("\nExample repeated orders:")
print(
    Order_data[
        Order_data["order_id"].duplicated(keep=False)
    ]
    .sort_values("order_id")
    .head(20)
)

Total order-item rows: 397482
Unique order IDs: 138116
Repeated order ID rows: 259366

Example repeated orders:
          order_id   product_id  quantity  unit_price  discount_percentage  \
167742  ORD-100002  PROD-000132         1      643.49             0.000000   
167743  ORD-100002  PROD-000730         4      233.33             0.000000   
264949  ORD-100004  PROD-000475         1       12.91             0.094303   
264950  ORD-100004  PROD-000912         1      335.86             0.042021   
264948  ORD-100004  PROD-000782         1      116.11             0.098960   
140726  ORD-100016  PROD-000319         2      176.23             0.180713   
140725  ORD-100016  PROD-001037         1       61.00             0.090087   
168896  ORD-100022  PROD-001140         1      199.76             0.153306   
168895  ORD-100022  PROD-000717         3      155.85             0.449832   
168897  ORD-100022  PROD-000381         2      416.72             0.186442   
267342  ORD-100023  PROD-00013

In [6]:
print(
    "Duplicate order-product pairs:",
    Order_data.duplicated(
        subset=["order_id", "product_id"]
    ).sum()
)
duplicate_pairs = Order_data[
    Order_data.duplicated(
        subset=["order_id", "product_id"],
        keep=False
    )
].sort_values(["order_id", "product_id"])

duplicate_pairs.head(20)

Duplicate order-product pairs: 0


,order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost,profit


In [7]:
duplicate_pairs = Order_data[
    Order_data.duplicated(
        subset=["order_id", "product_id"],
        keep=False
    )
].sort_values(["order_id", "product_id"])

print("Rows involved in duplicate pairs:", len(duplicate_pairs))

print(
    "Number of duplicate order-product combinations:",
    duplicate_pairs[["order_id", "product_id"]]
    .drop_duplicates()
    .shape[0]
)

print("Exact duplicate rows:", Order_data.duplicated().sum())

Rows involved in duplicate pairs: 0
Number of duplicate order-product combinations: 0
Exact duplicate rows: 0


In [8]:
# Expected gross sales before discount
expected_gross = Order_data["quantity"] * Order_data["unit_price"]

# Compare expected and recorded gross sales
gross_difference = (
    expected_gross - Order_data["gross_sales"]
).abs()

print(
    "Rows where gross sales do not match quantity × unit price:",
    (gross_difference > 0.02).sum()
)

# Identify rows with missing financial values
financial_cols = [
    "unit_price", "discount_amount", "gross_sales",
    "tax_amount", "shipping_cost", "net_sales",
    "product_cost", "profit"
]

print("\nMissing financial values:")
print(Order_data[financial_cols].isna().sum())

Rows where gross sales do not match quantity × unit price: 0

Missing financial values:
unit_price         0
discount_amount    0
gross_sales        0
tax_amount         0
shipping_cost      0
net_sales          0
product_cost       0
profit             0
dtype: int64


In [9]:
Order_data[
    (Order_data["order_id"] == "ORD-116240") &
    (Order_data["product_id"] == "PROD-001079")
]

,order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost,profit
292582,ORD-116240,PROD-001079,4,42.15,0.043355,7.31,168.6,11.29,3.67,176.25,52.04,120.54


In [10]:
duplicate_pairs = Order_data[
    Order_data.duplicated(
        subset=["order_id", "product_id"],
        keep=False
    )
].sort_values(["order_id", "product_id"])
duplicate_pairs.head(20)


,order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost,profit


In [11]:
Order_data = Order_data.drop_duplicates(subset=["order_id", "product_id"], keep="first")



output_file = Path().cwd().parent / "data" / "Cleaned_data" /"order_items_cleaned.csv"

Order_data.to_csv(output_file, index=False)


Duplicated = Order_data.duplicated(subset=["order_id", "product_id"], keep=False)


In [12]:
Ecommerce_data["customer_id"].isin(Customer_data["customer_id"]).all()
Order_data["order_id"].isin(Ecommerce_data["order_id"]).all()
Order_data["product_id"].isin(Product_data["product_id"]).all()

np.True_

## 3. Dimension tables

In [13]:
dim_customer = Customer_data.copy()
dim_product = (Product_data.drop_duplicates("product_id")
                       .rename(columns={"unit_price": "list_unit_price",
                                        "product_cost": "standard_cost"})
                       .reset_index(drop=True))

print(dim_customer.shape, dim_product.shape)


(25000, 11) (1175, 9)


In [14]:
order_dates = pd.to_datetime(Ecommerce_data["order_date"])
d = pd.date_range(order_dates.min(), order_dates.max(), freq="D")

dim_date = pd.DataFrame({"date": d})
dim_date["date_key"]     = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"]         = dim_date["date"].dt.year
dim_date["quarter"]      = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month_number"] = dim_date["date"].dt.month
dim_date["month_name"]   = dim_date["date"].dt.month_name()
dim_date["week_of_year"] = dim_date["date"].dt.isocalendar().week.astype(int)
dim_date["day_name"]     = dim_date["date"].dt.day_name()
dim_date["is_weekend"]   = dim_date["date"].dt.dayofweek >= 5

In [15]:
order_attr_cols = [
    "order_id", "customer_id", "order_date", "order_time",
    "order_status", "sales_channel", "customer_type",
    "payment_method", "payment_status", "currency",
    "shipping_method", "warehouse", "delivery_status",
    "return_status", "return_reason",
    "marketing_channel", "campaign_name", "coupon_code",
]

fact_sales = Order_data.merge(Ecommerce_data[order_attr_cols], on="order_id", how="left", validate="m:1")

fact_sales["order_date"] = pd.to_datetime(fact_sales["order_date"])
fact_sales["date_key"]   = fact_sales["order_date"].dt.strftime("%Y%m%d").astype(int)
fact_sales.insert(0, "order_item_key", np.arange(1, len(fact_sales) + 1))


print(fact_sales.shape)

print(fact_sales.columns)

fact_sales.head()



(397482, 31)
Index(['order_item_key', 'order_id', 'product_id', 'quantity', 'unit_price',
       'discount_percentage', 'discount_amount', 'gross_sales', 'tax_amount',
       'shipping_cost', 'net_sales', 'product_cost', 'profit', 'customer_id',
       'order_date', 'order_time', 'order_status', 'sales_channel',
       'customer_type', 'payment_method', 'payment_status', 'currency',
       'shipping_method', 'warehouse', 'delivery_status', 'return_status',
       'return_reason', 'marketing_channel', 'campaign_name', 'coupon_code',
       'date_key'],
      dtype='str')


,order_item_key,order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,...,currency,shipping_method,warehouse,delivery_status,return_status,return_reason,marketing_channel,campaign_name,coupon_code,date_key
0,1,ORD-301242,PROD-000738,2,244.40,0.339607,166.00,488.80,22.60,6.48,...,USD,Standard,WH-003,On Time,Not Returned,Not Applicable,Direct,Default_Campaign,No Coupon,20231106
1,2,ORD-301242,PROD-000181,3,287.13,0.362076,311.89,861.39,38.47,5.55,...,USD,Standard,WH-003,On Time,Not Returned,Not Applicable,Direct,Default_Campaign,No Coupon,20231106
2,3,ORD-773460,PROD-000973,4,754.80,0.468300,1413.89,3019.20,305.01,9.00,...,EUR,Economy,WH-005,On Time,Not Returned,Not Applicable,Direct,Default_Campaign,No Coupon,20251224
3,4,ORD-773460,PROD-000268,4,31.36,0.336182,42.17,125.44,15.82,0.00,...,EUR,Economy,WH-005,On Time,Not Returned,Not Applicable,Direct,Default_Campaign,No Coupon,20251224
4,5,ORD-449374,PROD-001040,3,6.31,0.089678,1.70,18.93,1.21,0.98,...,USD,Express,WH-015,On Time,Not Returned,Not Applicable,YouTube,Unknown Campaign,No Coupon,20210705


In [16]:
order_measure_cols = [
    "order_id", "customer_id", "order_date",
    "delivery_days", "estimated_delivery_days", "customer_rating",
    "review_sentiment", "customer_review",
    "loyalty_points_earned", "loyalty_points_redeemed",
    "profit_margin_percentage", "customer_lifetime_value",
    "is_repeat_customer", "customer_order_count",
]

fact_orders = Ecommerce_data[order_measure_cols].copy()
fact_orders["order_date"] = pd.to_datetime(fact_orders["order_date"])
fact_orders["date_key"]   = fact_orders["order_date"].dt.strftime("%Y%m%d").astype(int)
fact_orders["review_sentiment"] = fact_orders["review_sentiment"].fillna("No Review")

# keep NULLs but avoid floats like 3.0 for whole-number columns
for c in ["delivery_days", "estimated_delivery_days"]:
    fact_orders[c] = fact_orders[c].astype("Int16")

print(fact_orders.shape)
fact_orders.head()

(138116, 15)


,order_id,customer_id,order_date,delivery_days,estimated_delivery_days,customer_rating,review_sentiment,customer_review,loyalty_points_earned,loyalty_points_redeemed,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count,date_key
0,ORD-301242,CUST-003102,2023-11-06,3,3,3.5,Positive,Satisfied with the purchase.,94,44,36.50,12459.68,True,11,20231106
1,ORD-773460,CUST-003124,2025-12-24,10,10,3.6,Positive,Good value for money.,201,171,-1.11,13032.48,True,11,20251224
2,ORD-449374,CUST-012496,2021-07-05,3,3,4.3,Positive,Good product. Works as expected.,46,16,42.18,6159.44,True,6,20210705
3,ORD-567636,CUST-023928,2023-01-21,2,2,3.5,Positive,Satisfied with the purchase.,54,10,45.35,5638.30,True,7,20230121
4,ORD-820028,CUST-012730,2022-05-13,6,5,3.0,Neutral,Mixed feelings about this purchase.,278,147,54.94,13240.10,True,8,20220513


In [17]:
fact_sales = fact_sales.head(300000)

In [18]:
fact_sales["customer_id"].isin(dim_customer["customer_id"]).all()
fact_sales["product_id"].isin(dim_product["product_id"]).all()
fact_sales["order_date"].isin(dim_date["date"]).all()



save_dir = Path().cwd().parent / "data" / "data_modeling"


dim_customer.to_csv(save_dir / "dim_customer.csv", index=False)
dim_product.to_csv(save_dir / "dim_product.csv", index=False)
dim_date.to_csv(save_dir / "dim_date.csv", index=False)
fact_sales.to_csv(save_dir / "fact_sales.csv", index=False)
fact_orders.to_csv(save_dir / "fact_orders.csv", index=False)


In [19]:
dim_customer.info()
dim_product.info()
dim_date.info()
fact_sales.info()
fact_orders.info()


<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                25000 non-null  str    
 1   customer_name              25000 non-null  str    
 2   customer_age               25000 non-null  int64  
 3   gender                     25000 non-null  str    
 4   customer_segment           25000 non-null  str    
 5   customer_city              25000 non-null  str    
 6   customer_state             25000 non-null  str    
 7   customer_country           25000 non-null  str    
 8   region                     25000 non-null  str    
 9   customer_postal_code       25000 non-null  int64  
 10  customer_acquisition_cost  25000 non-null  float64
dtypes: float64(1), int64(2), str(8)
memory usage: 2.1 MB
<class 'pandas.DataFrame'>
RangeIndex: 1175 entries, 0 to 1174
Data columns (total 9 columns):
 #   Column          